# Mesh Visualization with topologic_fast

This notebook demonstrates mesh generation and visualization using `topologic_fast`.

We'll cover:
- Converting topologies to meshes
- Visualizing meshes with Plotly
- Exporting meshes to OBJ and STL formats
- Creating publication-quality 3D visualizations

**Note**: This notebook is adapted from the topologicpy mesh_plot tutorial. The original uses matplotlib;
this version uses Plotly for more interactive visualizations.

## Import Libraries

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print(f"topologic_fast version: {tf.__version__}")

## Mesh Basics

A mesh is a collection of vertices and triangular faces that approximate a 3D surface.
topologic_fast can generate meshes from any topology (Face, Shell, Cell, etc.).

In [ ]:
# Create a simple face and convert to mesh
face = tf.Face.Rectangle(width=2.0, length=3.0)
mesh = tf.Mesh.ByFace(face)

print(f"Face Mesh:")
print(f"  Vertices: {mesh.NumVertices()}")
print(f"  Triangles: {mesh.NumTriangles()}")
print(f"  Area: {mesh.Area():.4f}")

## Helper Functions for Visualization

In [ ]:
def parse_obj(obj_content):
    """Parse OBJ content into vertices and faces arrays."""
    vertices = []
    faces = []
    
    for line in obj_content.strip().split('\n'):
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == 'v':
            vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
        elif parts[0] == 'f':
            # OBJ faces are 1-indexed and may have texture/normal indices
            face_indices = [int(p.split('/')[0]) - 1 for p in parts[1:]]
            if len(face_indices) >= 3:
                faces.append(face_indices[:3])  # Take first 3 vertices
    
    return np.array(vertices), np.array(faces)


def mesh_to_plotly(mesh, color='lightblue', opacity=0.8, name='Mesh'):
    """Convert a topologic_fast Mesh to a Plotly Mesh3d trace."""
    obj_content = mesh.ToOBJ()
    vertices, faces = parse_obj(obj_content)
    
    if len(vertices) == 0 or len(faces) == 0:
        return None
    
    return go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=opacity,
        flatshading=True,
        name=name
    )


def cell_to_plotly(cell, color='lightblue', opacity=0.8, name='Cell'):
    """Convert a topologic_fast Cell to a Plotly Mesh3d trace."""
    mesh = tf.Mesh.ByCell(cell)
    return mesh_to_plotly(mesh, color, opacity, name)


def add_wireframe(fig, cell, color='black', width=1):
    """Add wireframe edges to a figure."""
    edges = cell.Edges()
    for edge in edges:
        start = edge.StartVertex()
        end = edge.EndVertex()
        fig.add_trace(go.Scatter3d(
            x=[start.X(), end.X()],
            y=[start.Y(), end.Y()],
            z=[start.Z(), end.Z()],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False,
            hoverinfo='skip'
        ))

## Visualizing Simple Shapes

In [ ]:
# Create various primitive shapes
box = tf.Cell.Box(0, 0, 0, 2, 2, 2)
cylinder = tf.Cell.Cylinder(5, 0, 0, 1, 2, 32)
sphere = tf.Cell.Sphere(10, 0, 0, 1, 16, 8)

# Create figure
fig = go.Figure()

# Add meshes
fig.add_trace(cell_to_plotly(box, color='coral', name='Box'))
fig.add_trace(cell_to_plotly(cylinder, color='lightgreen', name='Cylinder'))
fig.add_trace(cell_to_plotly(sphere, color='steelblue', name='Sphere'))

# Add wireframes
add_wireframe(fig, box)
add_wireframe(fig, cylinder)
add_wireframe(fig, sphere)

fig.update_layout(
    title='Simple Shapes as Meshes',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=900,
    height=600
)

fig.show()

## Torus Visualization

The torus is a classic test shape for mesh visualization.

In [ ]:
# Note: Torus may not be available in topologic_fast yet
# If not available, we'll create an approximation or use a different shape

try:
    # Try to create a torus if available
    torus = tf.Cell.Torus(0, 0, 0, 2.0, 0.5, 32, 16)
    has_torus = True
except AttributeError:
    # Torus not available - create an alternative
    print("Note: Cell.Torus() is not available in this version of topologic_fast.")
    print("Creating an alternative shape instead.")
    has_torus = False

if has_torus:
    mesh = tf.Mesh.ByCell(torus)
    
    print(f"Torus Mesh:")
    print(f"  Vertices: {mesh.NumVertices()}")
    print(f"  Triangles: {mesh.NumTriangles()}")
    print(f"  Area: {mesh.Area():.4f}")
    
    fig = go.Figure()
    fig.add_trace(mesh_to_plotly(mesh, color='gold', opacity=0.9, name='Torus'))
    
    fig.update_layout(
        title='Torus Mesh',
        scene=dict(
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1))
        ),
        width=800,
        height=600
    )
    
    fig.show()
else:
    # Create a complex shape from boolean operations instead
    outer = tf.Cell.Cylinder(0, 0, -0.25, 2, 0.5, 32)
    inner = tf.Cell.Cylinder(0, 0, -0.3, 1.5, 0.6, 32)
    ring = tf.Topology.Difference(outer, inner)
    
    mesh = tf.Mesh.ByCell(ring)
    
    print(f"Ring Mesh (torus approximation):")
    print(f"  Vertices: {mesh.NumVertices()}")
    print(f"  Triangles: {mesh.NumTriangles()}")
    
    fig = go.Figure()
    fig.add_trace(mesh_to_plotly(mesh, color='gold', opacity=0.9, name='Ring'))
    add_wireframe(fig, ring, color='darkgoldenrod', width=1)
    
    fig.update_layout(
        title='Ring Mesh (Torus Approximation)',
        scene=dict(aspectmode='data'),
        width=800,
        height=600
    )
    
    fig.show()

## OBJ Export

OBJ is a widely-supported 3D format for mesh data.

In [ ]:
# Create a mesh and export to OBJ
cell = tf.Cell.Box(0, 0, 0, 2, 2, 2)
mesh = tf.Mesh.ByCell(cell)

obj_content = mesh.ToOBJ()

print("OBJ Export Preview (first 30 lines):")
print("=" * 50)
lines = obj_content.strip().split('\n')
for i, line in enumerate(lines[:30]):
    print(line)

if len(lines) > 30:
    print(f"... ({len(lines) - 30} more lines)")

print(f"\nTotal: {len([l for l in lines if l.startswith('v ')])} vertices, "
      f"{len([l for l in lines if l.startswith('f ')])} faces")

## STL Export

STL is commonly used for 3D printing.

In [ ]:
# Export to STL
stl_content = mesh.ToSTL()

print("STL Export Preview (first 30 lines):")
print("=" * 50)
lines = stl_content.strip().split('\n')
for i, line in enumerate(lines[:30]):
    print(line)

if len(lines) > 30:
    print(f"... ({len(lines) - 30} more lines)")

## Complex Mesh from Boolean Operations

In [ ]:
# Create a complex shape using boolean operations
base = tf.Cell.Box(0, 0, 0, 4, 4, 1)

# Add pillars
p1 = tf.Cell.Cylinder(0.5, 0.5, 1, 0.4, 3, 16)
p2 = tf.Cell.Cylinder(3.5, 0.5, 1, 0.4, 3, 16)
p3 = tf.Cell.Cylinder(0.5, 3.5, 1, 0.4, 3, 16)
p4 = tf.Cell.Cylinder(3.5, 3.5, 1, 0.4, 3, 16)

# Roof
roof = tf.Cell.Box(0, 0, 4, 4, 4, 0.5)

# Combine
structure = base
for pillar in [p1, p2, p3, p4]:
    structure = tf.Topology.Union(structure, pillar)
structure = tf.Topology.Union(structure, roof)

# Subtract a window
window = tf.Cell.Box(1.5, -0.5, 2, 1, 1.5, 1.5)
structure = tf.Topology.Difference(structure, window)

# Create mesh
mesh = tf.Mesh.ByCell(structure)

print(f"Complex Structure Mesh:")
print(f"  Vertices: {mesh.NumVertices()}")
print(f"  Triangles: {mesh.NumTriangles()}")
print(f"  Area: {mesh.Area():.2f}")

In [ ]:
# Visualize the complex structure
fig = go.Figure()

fig.add_trace(mesh_to_plotly(mesh, color='sandybrown', opacity=0.9, name='Structure'))
add_wireframe(fig, structure, color='saddlebrown', width=1)

fig.update_layout(
    title='Complex Structure Mesh',
    scene=dict(
        aspectmode='data',
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1))
    ),
    width=900,
    height=700
)

fig.show()

## Mesh Quality Analysis

In [ ]:
# Analyze mesh quality for different shapes
shapes = [
    ('Box', tf.Cell.Box(0, 0, 0, 1, 1, 1)),
    ('Cylinder (16 sides)', tf.Cell.Cylinder(0, 0, 0, 1, 1, 16)),
    ('Cylinder (32 sides)', tf.Cell.Cylinder(0, 0, 0, 1, 1, 32)),
    ('Sphere (8x4)', tf.Cell.Sphere(0, 0, 0, 1, 8, 4)),
    ('Sphere (16x8)', tf.Cell.Sphere(0, 0, 0, 1, 16, 8)),
    ('Sphere (32x16)', tf.Cell.Sphere(0, 0, 0, 1, 32, 16)),
]

print("Mesh Quality Comparison:")
print("=" * 70)
print(f"{'Shape':<25} {'Vertices':>10} {'Triangles':>12} {'Area':>10}")
print("-" * 70)

for name, cell in shapes:
    mesh = tf.Mesh.ByCell(cell)
    print(f"{name:<25} {mesh.NumVertices():>10} {mesh.NumTriangles():>12} {mesh.Area():>10.4f}")

## Color by Position

Visualize a mesh with colors based on vertex position.

In [ ]:
# Create a sphere and color by Z position
sphere = tf.Cell.Sphere(0, 0, 0, 2, 32, 16)
mesh = tf.Mesh.ByCell(sphere)

obj_content = mesh.ToOBJ()
vertices, faces = parse_obj(obj_content)

# Color based on Z coordinate
z_values = vertices[:, 2]

fig = go.Figure(data=[go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    intensity=z_values,
    colorscale='Viridis',
    intensitymode='vertex',
    showscale=True,
    colorbar=dict(title='Z Height')
)])

fig.update_layout(
    title='Sphere Colored by Z Position',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=800,
    height=600
)

fig.show()

## Multiple Views

In [ ]:
# Create a shape to view from multiple angles
shape = tf.Cell.Box(0, 0, 0, 3, 2, 4)
mesh = tf.Mesh.ByCell(shape)
obj_content = mesh.ToOBJ()
vertices, faces = parse_obj(obj_content)

# Create subplots for different views
fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=['Front (XZ)', 'Side (YZ)', 'Top (XY)', 'Isometric']
)

# Add the same mesh to each subplot
for i in range(4):
    trace = go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color='steelblue',
        opacity=0.9,
        flatshading=True
    )
    fig.add_trace(trace, row=1, col=i+1)

# Set different camera angles
cameras = [
    dict(eye=dict(x=0, y=-2, z=0)),  # Front
    dict(eye=dict(x=2, y=0, z=0)),   # Side
    dict(eye=dict(x=0, y=0, z=2)),   # Top
    dict(eye=dict(x=1.5, y=-1.5, z=1.2))  # Isometric
]

for i, cam in enumerate(cameras):
    fig.update_scenes(camera=cam, row=1, col=i+1)
    fig.update_scenes(aspectmode='data', row=1, col=i+1)

fig.update_layout(
    title='Box from Multiple Views',
    width=1200,
    height=400
)

fig.show()

## Face to Mesh

In [ ]:
# Create different types of faces and visualize their meshes
faces_to_show = [
    ('Rectangle', tf.Face.Rectangle(width=3, length=2)),
    ('Circle', tf.Face.Circle(radius=1.5, sides=32)),
]

# Face with hole
outer_wire = tf.Wire.Rectangle(width=3, length=3)
inner_wire = tf.Wire.Rectangle(width=1, length=1)
face_with_hole = tf.Face.ByExternalInternalBoundaries(outer_wire, [inner_wire])
faces_to_show.append(('Face with Hole', face_with_hole))

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=[name for name, _ in faces_to_show]
)

colors = ['coral', 'lightgreen', 'steelblue']

for i, ((name, face), color) in enumerate(zip(faces_to_show, colors)):
    mesh = tf.Mesh.ByFace(face)
    obj_content = mesh.ToOBJ()
    vertices, faces_arr = parse_obj(obj_content)
    
    if len(vertices) > 0:
        fig.add_trace(go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces_arr[:, 0] if len(faces_arr) > 0 else [],
            j=faces_arr[:, 1] if len(faces_arr) > 0 else [],
            k=faces_arr[:, 2] if len(faces_arr) > 0 else [],
            color=color,
            opacity=0.9,
            flatshading=True
        ), row=1, col=i+1)
    
    fig.update_scenes(aspectmode='data', row=1, col=i+1)

fig.update_layout(
    title='Face Meshes',
    width=1200,
    height=400
)

fig.show()

# Print mesh statistics
print("\nFace Mesh Statistics:")
print("=" * 50)
for name, face in faces_to_show:
    mesh = tf.Mesh.ByFace(face)
    print(f"{name}: {mesh.NumVertices()} vertices, {mesh.NumTriangles()} triangles, area = {mesh.Area():.2f}")

## Saving Meshes to Files

In [ ]:
# Example of saving mesh to file (not actually saving to avoid file system changes)
cell = tf.Cell.Sphere(0, 0, 0, 1, 32, 16)
mesh = tf.Mesh.ByCell(cell)

obj_content = mesh.ToOBJ()
stl_content = mesh.ToSTL()

print("To save to files, use:")
print("")
print("# OBJ format")
print("with open('sphere.obj', 'w') as f:")
print("    f.write(mesh.ToOBJ())")
print("")
print("# STL format")
print("with open('sphere.stl', 'w') as f:")
print("    f.write(mesh.ToSTL())")
print("")
print(f"OBJ size: {len(obj_content)} characters")
print(f"STL size: {len(stl_content)} characters")

## Note on Matplotlib

The original topologicpy tutorial uses matplotlib for visualization. Here's how you could
use matplotlib if preferred (requires matplotlib to be installed):

In [ ]:
# Matplotlib alternative (uncomment to use)
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection
#
# def plot_mesh_matplotlib(mesh, title='Mesh'):
#     obj_content = mesh.ToOBJ()
#     vertices, faces = parse_obj(obj_content)
#
#     fig = plt.figure(figsize=(10, 8))
#     ax = fig.add_subplot(111, projection='3d')
#
#     # Create polygon collection
#     face_vertices = [[vertices[idx] for idx in face] for face in faces]
#     poly3d = Poly3DCollection(face_vertices, facecolors='cyan', linewidths=1, edgecolors='black', alpha=0.7)
#     ax.add_collection3d(poly3d)
#
#     # Set axis limits
#     ax.set_xlim([vertices[:, 0].min(), vertices[:, 0].max()])
#     ax.set_ylim([vertices[:, 1].min(), vertices[:, 1].max()])
#     ax.set_zlim([vertices[:, 2].min(), vertices[:, 2].max()])
#
#     ax.set_xlabel('X')
#     ax.set_ylabel('Y')
#     ax.set_zlabel('Z')
#     ax.set_title(title)
#
#     plt.show()
#
# # Usage:
# # cell = tf.Cell.Box(0, 0, 0, 2, 2, 2)
# # mesh = tf.Mesh.ByCell(cell)
# # plot_mesh_matplotlib(mesh, 'Box Mesh')

print("Matplotlib example code is provided above (commented out).")
print("Uncomment and run if you prefer matplotlib over Plotly.")

## Summary

This notebook demonstrated:

### Mesh Creation
- `tf.Mesh.ByFace(face)` - Create mesh from a face
- `tf.Mesh.ByCell(cell)` - Create mesh from a cell

### Mesh Properties
- `mesh.NumVertices()` - Get vertex count
- `mesh.NumTriangles()` - Get triangle count
- `mesh.Area()` - Get total surface area

### Mesh Export
- `mesh.ToOBJ()` - Export to OBJ format
- `mesh.ToSTL()` - Export to STL format

### Visualization
- Plotly Mesh3d for interactive 3D visualization
- Color mapping by position or other properties
- Multiple view angles
- Wireframe overlays

### Applications
- 3D printing (STL export)
- Game/visualization assets (OBJ export)
- Surface area calculations
- Mesh quality analysis